In [3]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# -----------------------------
# 1️⃣ Load Data
# -----------------------------
train_df = pd.read_csv("traindata.csv")
test_df = pd.read_csv("testwithlabels.csv")

train_df = train_df.dropna()
test_df = test_df.dropna()

train_df['Class'] = train_df['Class'].str.strip().str.lower()
test_df['Class'] = test_df['Class'].str.strip().str.lower()

le = LabelEncoder()
train_df["Class"] = le.fit_transform(train_df["Class"])
test_df["Class"] = le.transform(test_df["Class"])

num_labels = len(le.classes_)

# -----------------------------
# 2️⃣ Class Weights (for imbalance)
# -----------------------------
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df["Class"]),
    y=train_df["Class"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

# -----------------------------
# 3️⃣ Tokenizer
# -----------------------------
model_name = "xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)

class TamilDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=max_len
        )
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TamilDataset(train_df["Text"], train_df["Class"], tokenizer)
test_dataset = TamilDataset(test_df["Text"], test_df["Class"], tokenizer)

# -----------------------------
# 4️⃣ Model
# -----------------------------
model = XLMRobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# -----------------------------
# 5️⃣ Custom Weighted Trainer
# -----------------------------
class WeightedTrainer(Trainer):
    # FIX: Added `num_items_in_batch` to the compute_loss signature
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=0):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# -----------------------------
# 6️⃣ Metrics
# -----------------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# -----------------------------
# 7️⃣ Training Arguments (3 EPOCHS)
# -----------------------------
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_dir="./logs",
)

# -----------------------------
# 8️⃣ Trainer
# -----------------------------
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# -----------------------------
# 9️⃣ Train & Evaluate
# -----------------------------
trainer.train()
results = trainer.evaluate()

print("Final Test Results:", results)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.670852,0.634063,0.687842,0.686221,0.696161,0.687842
2,0.592153,0.597840,0.714129,0.711105,0.719074,0.714129
3,0.500952,0.584690,0.733844,0.733759,0.733761,0.733844


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Final Test Results: {'eval_loss': 0.5846904516220093, 'eval_accuracy': 0.7338444687842278, 'eval_f1': 0.7337593885723565, 'eval_precision': 0.733760675281219, 'eval_recall': 0.7338444687842278, 'eval_runtime': 1.9203, 'eval_samples_per_second': 475.443, 'eval_steps_per_second': 59.886, 'epoch': 3.0}
